# 📝 과제 01: 온라인 서점 상담 챗봇 모델 고르고 실행하기

## 만들 결과

온라인 서점 고객센터가 주문·배송 문의에 답할 오픈 웨이트 LLM을 고르고, 교안 01과 같은 순서로 실행합니다. 고객의 주문 정보는 밖으로 보낼 수 없어 로컬 GPU(24GB)에서 실행하고, 상업 서비스라 라이선스를 확인합니다.

<img src="images/과제/40_gwaje01_flow.png" width="1050" alt="제목은 '서점 상담 챗봇 모델 고르고 실행하기'입니다. 카드 다섯 장이 왼쪽에서 오른쪽으로 화살표로 이어집니다. 1 MoE 읽기(전체 · 활성 파라미터), 2 후보 표(7개 -> 2개), 3 설정 읽기(config · 생성 기본값), 4 Transformers 답(greedy · 샘플링), 5 Ollama 답(LangChain · 도구 호출). 카드 아래에 노트북을 든 개발자가 있습니다.">

- 모델 이름과 설정 파일에서 MoE의 전체·활성 크기와 전문가 수를 읽습니다.
- 후보 7개의 후보 표를 만들어 조건 네 개로 거르고, 남은 모델 가운데 가장 작은 모델을 고릅니다.
- 고른 모델의 설정과, 실행 모델이 Transformers와 Ollama에서 쓰는 생성 기본값을 확인합니다.
- 서점 상담 질문에 추론 모드를 끄고 Transformers와 Ollama로 답을 받습니다.
- LangChain으로 문의를 정해진 형식으로 분류하고, 주문 조회 도구를 불러 답하게 합니다.

## 준비: 모델과 제공 코드

- **실행 모델:** 교안 01과 같은 `Qwen/Qwen3.5-4B`(Transformers)와 Ollama `qwen3.5:4b-q4_K_M`입니다. Ollama 서버가 켜져 있어야 합니다. 2번 후보들은 Hub 정보와 설정 파일만 조회합니다.
- **맨 위 셀:** 실습 모델 파일을 미리 내려받습니다. 커널을 다시 시작할 때도 이 셀부터 실행합니다(받은 파일은 확인만 합니다).
- **시작 전:** 앞 노트북의 커널을 종료(Shut Down)합니다. 준비 셀이 Ollama가 GPU에 올려 둔 모델을 내립니다.
- **제공 코드:** 준비 셀, 교안 01의 `candidate_row`, 모델 불러오기, 6번의 분류 클래스와 도구 함수
- **작성:** `[작성]` 셀의 `...`를 채우고 `[자가채점]` 셀을 실행하세요. 뒤 문항이 앞 문항의 변수를 쓰므로 1번부터 차례로 풉니다.

In [ ]:
import asyncio

from huggingface_hub import snapshot_download
import ollama

# 실습에서 쓸 모델을 미리 내려받습니다. 처음 한 번만 오래 걸리고, 다음부터는 이미 받은 파일을 확인만 합니다
# Transformers용 원본 모델(BF16, 약 9GB): Hugging Face 캐시 폴더(기본 ~/.cache/huggingface/hub)에 저장됩니다
# snapshot_download는 일반 함수라서 asyncio.to_thread로 감싸야 다른 다운로드와 함께 기다릴 수 있습니다
tasks = [asyncio.to_thread(snapshot_download, "Qwen/Qwen3.5-4B")]

# Ollama용 모델(GGUF 파일): pull 요청을 받은 Ollama 서버가 내려받아 서버의 모델 폴더(기본 ~/.ollama/models)에 보관합니다
# 태그 이름은 ollama.com/search 에서 모델을 검색하고 모델 페이지의 Tags 탭에서 고릅니다
ollama_client = ollama.AsyncClient()
ollama_tags = ["qwen3.5:4b-q4_K_M"]
for tag in ollama_tags:
    tasks.append(ollama_client.pull(tag))

# 두 다운로드는 받는 곳과 저장 위치가 달라 서로 기다릴 필요가 없으므로 동시에 받고, 결과는 tasks 순서대로 받습니다
results = await asyncio.gather(*tasks)
model_dir = results[0]
print("Hugging Face 모델:", model_dir)
print("Ollama 모델:", ollama_tags)

In [ ]:
import pandas as pd

import torch

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다 (지난 단원과 같은 코드)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

from huggingface_hub import HfApi
from transformers import AutoConfig

# Hub 조회 도구(candidate_row가 씁니다)
api = HfApi()

# 표에서 긴 답이 잘리지 않게 합니다
pd.set_option("display.max_colwidth", None)

In [ ]:
# 교안 01에서 만든 후보 표의 한 행을 만드는 함수입니다
def candidate_row(repo_id):
    """모델 저장소 정보를 조회해 후보 표의 한 행(dict)으로 돌려줍니다."""
    # 가중치는 받지 않고 모델 카드, safetensors 요약, 게이트 정보만 조회합니다
    info = api.model_info(repo_id, expand=["cardData", "safetensors", "gated", "pipeline_tag"])

    # 파라미터 수는 safetensors 요약의 total, 저장 정밀도는 parameters의 키(예: BF16)입니다
    total_params = info.safetensors.total

    # 16비트(2바이트)로 불러올 때 가중치 메모리를 GiB(1024의 세제곱 바이트)로 계산합니다
    bf16_gib = total_params * 2 / 1024**3
    return {
        "모델": repo_id,
        "라이선스": info.card_data.license,
        "라이선스 이름": info.card_data.license_name,
        "게이트": info.gated,
        "하는 일": info.pipeline_tag,
        "파라미터 수": total_params,
        "저장 정밀도": list(info.safetensors.parameters),
        "BF16 가중치 GiB": round(bf16_gib, 2),
    }

In [ ]:
# 앞에서 Ollama가 GPU에 올려 둔 모델이 있으면 내립니다. keep_alive=0은 "지금 바로 GPU에서 내리기"라는 뜻입니다
for loaded in ollama.ps().models:
    ollama.generate(model=loaded.model, keep_alive=0)
print("Ollama가 GPU에 올린 모델:", ollama.ps().models)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3.5-4B"

# 토크나이저를 불러옵니다. 파일은 맨 위 미리 내려받기 셀이 받아 두었으므로 캐시에서 읽습니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Qwen3.5-4B는 이미지도 받는 모델이지만, AutoModelForCausalLM은 글 생성 부분만 불러옵니다
# 모델 파일에 저장된 정밀도(BF16) 그대로 계산 장치에 올립니다(GPU 메모리 약 8GB). 불러온 모델은 이미 평가 모드입니다
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16).to(device)
print("불러온 모델:", MODEL_NAME, "| 정밀도:", model.dtype, "| 장치:", model.device)

## 1. 이름과 설정으로 MoE 읽기

**배경**: MoE 모델은 이름의 `숫자B-A숫자B`가 전체·활성 파라미터 수입니다(교안 01 2-3, 3-1). 전문가 수는 설정 파일에 있는데, 이 모델은 글만 받는 모델이라 교안 01 2-3의 Qwen3.5와 달리 설정이 `text_config` 없이 맨 위에 있습니다.

**요구사항**
- (1) 문자열 `moe_name`에서 `/` 뒤의 모델 이름을 `-`로 나눠, `B`로 끝나고 `A`로 시작하지 않는 조각을 **`name_total`** 에, `A`로 시작하는 조각에서 `A`를 뗀 것을 **`name_active`** 에 담습니다(둘 다 문자열).
- (2) 같은 저장소의 설정 파일을 읽어 층마다 전문가 수를 정수 **`n_experts`**, 토큰마다 고르는 전문가 수를 정수 **`n_active_experts`** 에 담습니다.
- (2) `n_active_experts / n_experts`를 **`expert_ratio`** 에 담습니다.

**확인 기준**: 두 크기가 `B`로 끝나는 이름 조각이고, 전문가 수가 정수이며 고르는 수가 전체보다 적고, `expert_ratio`가 두 수의 비율입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 저장소 이름은 '/' 뒤가 모델 이름입니다. 모델 이름을 '-'로 나눈 조각들을 하나씩 보며 조건에 맞는 조각을 고릅니다.
- 설정 파일은 교안 01 2-3과 같은 방법으로 읽고, 설정 이름도 같습니다. text_config를 거치지 않습니다.

세부구현:
1. 조각이 'B'로 끝나는지, 'A'로 시작하는지 문자열 메서드로 확인합니다. 'Qwen3'처럼 B로 끝나지 않는 조각은 건너뜁니다.
2. 설정을 출력해 전문가 수를 담은 이름을 확인한 뒤 값을 꺼냅니다.
```

</details>

In [ ]:
# [작성] (1) 모델 이름에서 전체 크기와 활성 크기를 읽으세요.
moe_name = "Qwen/Qwen3-30B-A3B-Instruct-2507"

name_total = ...
name_active = ...

print("전체:", name_total, "| 활성:", name_active)

In [ ]:
# [작성] (2) 설정 파일에서 전문가 수를 읽으세요.
moe_config = ...

n_experts = ...
n_active_experts = ...
expert_ratio = ...

print("층마다 전문가 수:", n_experts, "| 토큰마다 고르는 전문가 수:", n_active_experts, "| 비율:", expert_ratio)

In [ ]:
# [자가채점] 이름에서 읽은 크기와 설정 파일의 전문가 수를 확인합니다.
name_parts = moe_name.split("/")[1].split("-")
assert isinstance(name_total, str) and name_total in name_parts and name_total.endswith("B") and not name_total.startswith("A"), f"name_total이 {name_total!r}입니다. '-'로 나눈 조각 가운데 B로 끝나고 A로 시작하지 않는 것을 고르세요."
assert isinstance(name_active, str) and "A" + name_active in name_parts, f"name_active가 {name_active!r}입니다. A로 시작하는 조각에서 맨 앞 A를 떼세요."
assert isinstance(n_experts, int) and isinstance(n_active_experts, int), "전문가 수는 설정 파일의 정수 값입니다. text_config를 거치지 않고 맨 위에서 읽으세요."
assert 1 < n_active_experts < n_experts, "n_active_experts(토큰마다 고르는 수)는 n_experts(층마다 전문가 수)보다 적습니다. 두 설정 이름을 바꿔 넣지 않았는지 확인하세요."
assert abs(expert_ratio - n_active_experts / n_experts) < 1e-9, "expert_ratio는 n_active_experts / n_experts입니다."

# 어텐션처럼 모든 토큰이 거치는 층이 있어, 전문가 비율은 이름의 활성/전체 비율보다 작습니다
name_ratio = float(name_active[:-1]) / float(name_total[:-1])
assert expert_ratio < name_ratio, "expert_ratio가 이름의 활성/전체 비율보다 큽니다. 설정 이름을 다시 확인하세요."
print("✅ 1번 통과")

**결과 읽기**
- **전문가 비율과 이름의 비율:** 토큰 하나는 층마다 고른 전문가만 거치므로 `expert_ratio`가 작습니다. 여기에 어텐션·임베딩처럼 모든 토큰이 함께 쓰는 층이 더해져, 실제 활성 비율(이름의 활성/전체)은 `expert_ratio`보다 큽니다.
- **설정 위치:** 같은 회사 모델이라도 이미지까지 받는 모델(Qwen3.5)은 `text_config` 안에, 글만 받는 모델(Qwen3)은 맨 위에 설정이 있습니다. 설정을 출력해 보고 꺼냅니다.

## 2. 후보 표와 거르기

**배경**: 서점 서비스는 상업 사용이고 GPU는 24GB 한 장입니다. 교안 01 3-4처럼 생성할 때 더 드는 20%까지 더해 24GB에 들어가야 합니다(가중치는 약 20GiB 이하). 지금 코드(`AutoModelForCausalLM`)로 글을 생성할 수 있는 모델이어야 합니다.

**요구사항**
- (1) 제공한 `book_ids`의 후보 7개를 교안 01의 `candidate_row`로 조회해 DataFrame **`book_candidates`**(7행)를 만듭니다.
- (2) 아래 조건을 **모두** 만족하는 행만 남긴 DataFrame **`book_ok`** 를 만듭니다.
  - `라이선스` 열이 `"apache-2.0"` 또는 `"mit"`
  - `게이트` 열이 `False`(이 열에는 `False`와 문자열 `"auto"`가 섞여 있으므로 `False`와 같은지 비교합니다)
  - `하는 일` 열이 `"text-generation"` 또는 `"image-text-to-text"`(이미지도 받는 모델이지만 `AutoModelForCausalLM`이 글 생성 부분만 불러오는 모델. 이 후보 목록에서는 Qwen3.5가 해당합니다). 이 클래스로 불러도 이미지·음성 인코더까지 함께 올라오는 `"any-to-any"`는 뺍니다.
  - `BF16 가중치 GiB` × 1.2가 24 이하
- (2) `book_ok` 가운데 `파라미터 수`가 가장 적은 모델의 저장소 전체 이름(예: `"trillionlabs/..."`)을 문자열 **`book_pick`** 에 담습니다.

**확인 기준**: `book_candidates`가 7행이고, `book_ok`의 모든 행이 네 조건을 만족하며 2행이고, `book_pick`이 `book_ok` 안에서 파라미터 수가 가장 적은 모델입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 후보 표는 교안 01 3-4와 같은 방법으로 만들고, 거르기는 교안 01 함께 따라하기 6-1처럼 조건마다 행별 True/False 값을 만들어 &로 묶습니다.

세부구현:
1. 라이선스 조건은 값이 목록 안에 있는지 확인하는 메서드(isin)를 씁니다.
2. 하는 일 조건도 라이선스처럼 값이 목록 안에 있는지 확인합니다. 메모리 조건은 BF16 가중치 GiB 열에 1.2를 곱해 24와 비교합니다.
3. 남은 표를 파라미터 수 기준으로 정렬하거나 가장 작은 값의 행을 찾아 모델 열의 값을 꺼냅니다.
```

</details>

In [ ]:
# [작성] (1) 후보 7개로 후보 표를 만드세요.
book_ids = [
    "trillionlabs/Tri-7B",
    "Qwen/Qwen3.5-2B",
    "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
    "AlmawaveLabs/Velvet-2B",
    "google/gemma-4-E2B-it",
    "microsoft/phi-4",
    "google/gemma-4-12b-it",
]

book_candidates = ...
display(book_candidates)

In [ ]:
# [작성] (2) 조건 네 개로 거르고 가장 작은 모델을 고르세요.
book_ok = ...
display(book_ok)

book_pick = ...
print("고른 모델:", book_pick)

In [ ]:
# [자가채점] 후보 표의 크기, 남은 행이 조건을 모두 만족하는지, 가장 작은 모델을 골랐는지 확인합니다.
assert isinstance(book_candidates, pd.DataFrame) and len(book_candidates) == 7, "book_candidates는 후보 7개의 candidate_row 결과로 만든 7행 DataFrame입니다."
assert set(book_candidates["모델"]) == set(book_ids), "book_candidates의 모델 열이 book_ids와 같아야 합니다."
assert isinstance(book_ok, pd.DataFrame) and len(book_ok) > 0, "book_ok는 조건을 만족하는 행만 남긴 DataFrame입니다."
for _, row in book_ok.iterrows():
    assert row["라이선스"] in ["apache-2.0", "mit"], f"{row['모델']}의 라이선스가 {row['라이선스']}입니다. 라이선스 조건을 확인하세요."
    assert row["게이트"] == False, f"{row['모델']}은 게이트가 있습니다."
    assert row["하는 일"] in ["text-generation", "image-text-to-text"], f"{row['모델']}은 {row['하는 일']} 모델입니다. 하는 일 조건을 확인하세요."
    assert row["BF16 가중치 GiB"] * 1.2 <= 24, f"{row['모델']}은 생성 시 메모리가 24 GiB를 넘습니다. 메모리 조건에 1.2를 곱했는지 확인하세요."
assert len(book_ok) == 2, f"book_ok가 {len(book_ok)}행입니다. 네 조건을 모두 &로 묶었는지 확인하세요(조건 하나로만 떨어지는 후보가 조건마다 있습니다)."
assert isinstance(book_pick, str) and "/" in book_pick, "book_pick은 'trillionlabs/...'처럼 저장소 전체 이름(문자열)입니다."
assert book_pick in set(book_ok["모델"]), "book_pick은 book_ok에 남은 모델 가운데 하나입니다."
assert book_ok.set_index("모델").loc[book_pick, "파라미터 수"] == book_ok["파라미터 수"].min(), "book_pick은 book_ok 가운데 파라미터 수가 가장 적은 모델입니다."
print("✅ 2번 통과")

**결과 읽기**
- **떨어진 이유:** EXAONE 3.5 7.8B(한국 LG)는 비상업 라이선스(`other`, `exaone`), Velvet-2B(이탈리아 Almawave)는 게이트(`auto`, 약관에 동의하면 바로 승인), Gemma 4 E2B는 이미지·음성 인코더까지 함께 올라오는 `any-to-any`, phi-4는 생성 시 약 32.8 GiB라 24GB를 넘어 떨어집니다. Gemma 4 12B는 `any-to-any`이고 생성 시 약 26.7 GiB라 두 조건에서 떨어집니다.
- **남은 두 모델:** `Qwen/Qwen3.5-2B`(약 22.7억 개, Apache-2.0, 중국 알리바바)와 `trillionlabs/Tri-7B`(약 75.3억 개, Apache-2.0, 한국 트릴리온랩스)입니다. 더 작은 Qwen3.5-2B를 고릅니다. 실습 모델 Qwen3.5-4B와 같은 계열의 작은 크기이고, 파라미터 수에는 이미지 처리 부분까지 들어 있습니다.
- **`image-text-to-text`는 모델마다 확인:** Qwen3.5는 `AutoModelForCausalLM`이 글 생성 부분만 불러오지만, 이미지도 받는 모델이 모두 그런 것은 아닙니다(예: Qwen3-VL은 이 클래스로 불러오지 못합니다). 모델 카드의 사용 예와 설정 파일로 확인합니다.
- **저장 정밀도 `F32`:** Tri-7B처럼 파일이 32비트로 저장된 모델도 `BF16 가중치 GiB` 열은 16비트로 불러올 때를 기준으로 계산한 값입니다.

## 3. 고른 모델의 설정과 생성 기본값

**배경**: 고른 모델을 내려받기 전에 설정 파일로 구조와 컨텍스트 길이를 확인합니다(교안 01 3-5). 그리고 실행 모델 Qwen3.5-4B가 두 실행 방법에서 어떤 생성 기본값을 쓰는지 확인합니다(교안 01 4-2, 5-4). Qwen3.5-4B 저장소에는 생성 기본값 파일(`generation_config.json`)이 없어 Transformers는 기본으로 greedy로 생성하고, Ollama는 모델 파일에 정해 둔 샘플링 값을 씁니다.

**요구사항**
- (1) `book_pick`의 설정 파일을 읽어 dict **`book_spec`** 을 만듭니다. 키는 `"층 수"`, `"은닉 크기"`, `"어휘 수"`, `"컨텍스트 길이"`이고 값은 설정의 정수입니다. 고른 모델은 이미지도 받는 모델이라 글 생성 부분 설정이 `text_config` 안에 있습니다(교안 01 2-3).
- (2) 제공 코드에서 불러온 `model`의 `generation_config`에서 `do_sample` 값을 **`book_hf_sampling`** 에 담습니다.
- (2) `ollama.show("qwen3.5:4b-q4_K_M")`의 `parameters`(기본 생성 설정 문자열)를 줄마다 `이름 값`으로 나눠, 이름을 키로, 값을 실수(`float`)로 담은 dict **`book_gen_defaults`** 를 만듭니다.

**확인 기준**: `book_spec`의 키가 위와 같고 값이 설정 파일과 같습니다. `book_hf_sampling`이 샘플링을 켜지 않은 값(`False` 또는 `None`)이고, `book_gen_defaults`가 Ollama 기본 설정 네 개를 담습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 설정 파일은 교안 01 3-5와 같은 방법으로 읽고, 값은 1번과 달리 text_config 안에서 꺼냅니다.
- parameters는 한 줄에 '이름   값'이 공백으로 이어진 여러 줄 문자열입니다. 먼저 print로 모양을 봅니다.

세부구현:
1. 문자열을 줄 단위로 나눈 뒤, 줄마다 공백으로 나눠 이름과 값을 꺼냅니다.
2. 값은 문자열이므로 float로 바꿔 dict에 넣습니다.
```

</details>

In [ ]:
# [작성] (1) 고른 모델의 설정에서 네 값을 dict로 모으세요.
book_config = ...

book_spec = {
    "층 수": ...,
    "은닉 크기": ...,
    "어휘 수": ...,
    "컨텍스트 길이": ...,
}
display(book_spec)

In [ ]:
# [작성] (2) Transformers와 Ollama의 생성 기본값을 확인하세요.
book_hf_sampling = ...
print("Transformers do_sample:", book_hf_sampling)

book_gen_defaults = ...
display(book_gen_defaults)

In [ ]:
# [자가채점] 두 dict의 키와 값을 확인합니다.
assert list(book_spec) == ["층 수", "은닉 크기", "어휘 수", "컨텍스트 길이"], "book_spec의 키를 지문과 같은 이름, 같은 순서로 쓰세요."
assert book_spec["은닉 크기"] != 2560, "book_spec은 2번에서 고른 모델의 설정입니다. 실행 모델(Qwen3.5-4B, 은닉 크기 2,560)의 설정이 아닙니다."
assert [book_spec[k] for k in book_spec] == [24, 2048, 248320, 262144], "book_pick 설정 파일의 text_config에서 층 수, 은닉 크기, 어휘 수, 컨텍스트 길이를 꺼냈는지 확인하세요."

assert not book_hf_sampling, "book_hf_sampling은 model.generation_config.do_sample 값입니다. 생성 기본값 파일이 없어 샘플링이 켜져 있지 않아야 합니다."
assert book_gen_defaults == {"presence_penalty": 1.5, "temperature": 1.0, "top_k": 20.0, "top_p": 0.95}, "book_gen_defaults는 ollama.show('qwen3.5:4b-q4_K_M').parameters를 줄마다 '이름 값'으로 나눠 값을 float로 담은 dict입니다."
print("✅ 3번 통과")

**결과 읽기**
- **층 24, 은닉 크기 2,048:** 실행 모델 Qwen3.5-4B(층 32, 은닉 크기 2,560)보다 층이 8개 적고 토큰 벡터가 짧습니다.
- **어휘 수 248,320, 컨텍스트 길이 262,144:** 같은 Qwen3.5 계열이라 실행 모델과 같습니다. 크기만 다르고 토크나이저와 한 번에 넣을 수 있는 길이는 같습니다.
- **두 실행 방법의 기본값이 다릅니다:** Transformers는 생성 기본값 파일이 없어 greedy로, Ollama는 temperature 1·top_p 0.95·top_k 20·presence_penalty 1.5(이미 나온 토큰이 다시 나오기 어렵게 하는 값)의 샘플링으로 생성합니다. 그래서 두 방법의 답을 비교하거나 고객에게 같은 안내를 하려면 4·5번처럼 생성 설정을 직접 적습니다.

## 4. 서점 상담 답 만들기

**배경**: 모델을 새로 내려받는 시간을 줄이려고 실행은 교안 01에서 받은 Qwen3.5-4B로 합니다. chat template은 모델마다 다르지만 `apply_chat_template`이 그 모델의 형식으로 문자열을 만들어 주므로, 2번에서 고른 Qwen3.5-2B로 바꿀 때도 모델 이름만 바꿔 같은 순서로 실행합니다. `enable_thinking`은 Qwen3.5 템플릿이 읽는 인자라, 이 인자를 쓰지 않는 다른 계열 모델의 템플릿에서는 무시됩니다.

**요구사항**
- (1) 제공한 `book_system`, `book_question`으로 system·user 메시지 목록 **`book_messages`** 를 만듭니다.
- (1) `book_messages`를 추론 모드를 끈 chat template 문자열 **`book_prompt`**(`tokenize=False`, `add_generation_prompt=True`, `enable_thinking=False`)로 바꾸고, `book_prompt`를 토크나이저에 넣어 나온 토큰 번호 목록의 길이를 정수 **`book_prompt_tokens`** 에 담습니다.
- (2) 질문 하나를 받아 `book_system` 역할로 추론 모드를 끄고 greedy(`do_sample=False`), 새 토큰 100개까지 생성해 **새 토큰만** 글자로 바꾼 문자열을 돌려주는 함수 **`ask_book(question)`** 을 만들고, `book_question`의 답을 **`book_answer`** 에 담습니다.
- (3) `torch.manual_seed(0)`을 한 번 부른 뒤, `book_prompt`의 토큰을 입력으로 모델 카드가 추론 모드를 끈 대화에 권하는 샘플링(`do_sample=True`, `temperature=0.7`, `top_p=0.8`, `top_k=20`, 새 토큰 60개)을 세 번 해 새 토큰만 글자로 바꾼 문자열 세 개를 리스트 **`book_samples`** 에 담습니다.

**확인 기준**: `book_prompt`가 system으로 시작하고 assistant 차례 표식 뒤의 빈 추론 칸(`<think>` `</think>`)으로 끝나며, `book_prompt_tokens`가 그 토큰 수입니다. `ask_book`이 특수 토큰과 추론 과정 없는 문자열을 돌려주고 같은 질문에 두 번 같은 답을 내며, `book_samples`가 문자열 3개이고 서로 다른 답이 2개 이상입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01 5-2~5-5의 순서(chat template -> 토큰 -> generate -> 새 토큰만 decode)를 함수 하나로 묶습니다. 교안의 chat 함수와 모양이 같고 system과 새 토큰 수가 다릅니다.

세부구현:
1. 토큰 수는 문자열을 토크나이저에 넣어 나온 토큰 번호 목록의 길이입니다.
2. 새 토큰만 남기려면 generate 결과에서 입력 길이 뒤만 잘라 냅니다.
3. 샘플링은 for문 세 번 안에서 generate를 부르고, 시드는 for문 밖에서 한 번만 고정합니다.
```

</details>

In [ ]:
# [작성] (1) 메시지와 chat template 문자열을 만들고 토큰 수를 세세요.
book_system = "당신은 온라인 서점 고객센터 상담원입니다. 고객의 질문에 한국어로 짧고 정중하게 답합니다."
book_question = "주문한 책이 3일째 배송 준비 중이에요. 언제 받을 수 있나요?"

book_messages = ...
book_prompt = ...
book_prompt_tokens = ...

print(book_prompt)
print("토큰 수:", book_prompt_tokens)

In [ ]:
# [작성] (2) 서점 상담원 역할로 greedy 답을 돌려주는 함수를 만드세요.
def ask_book(question):
    """서점 상담원 역할로 질문 하나에 greedy로 답한 문자열(새 토큰만)을 돌려줍니다."""
    ...


book_answer = ask_book(book_question)
print(book_answer)

In [ ]:
# [작성] (3) 시드를 한 번 고정하고 샘플링으로 세 번 생성하세요.
book_samples = ...

display(pd.DataFrame({"샘플링 답": book_samples}))

In [ ]:
# [자가채점] chat template 문자열, 토큰 수, 함수의 결정성, 샘플링 결과를 확인합니다.
assert isinstance(book_messages, list) and [m["role"] for m in book_messages] == ["system", "user"], "book_messages는 system, user 순서의 메시지 두 개입니다."
assert isinstance(book_prompt, str) and book_prompt.startswith("<|im_start|>system") and book_system in book_prompt, "book_prompt는 apply_chat_template으로 만든 문자열이고 system 메시지로 시작합니다. tokenize=False를 썼는지 확인하세요."
assert "<|im_start|>assistant\n" in book_prompt, "book_prompt에 assistant 차례 표식이 없습니다. add_generation_prompt=True를 썼는지 확인하세요."
assert book_prompt.endswith("<think>\n\n</think>\n\n"), "book_prompt 끝에 빈 추론 칸(<think> </think>)이 없습니다. enable_thinking=False를 넣어 추론 모드를 껐는지 확인하세요."
assert isinstance(book_prompt_tokens, int) and book_prompt_tokens == 58, f"book_prompt_tokens가 {book_prompt_tokens}입니다. book_prompt를 토크나이저에 넣어 나온 토큰 번호 목록의 길이(정수)를 담으세요."

assert isinstance(book_answer, str) and len(book_answer.strip()) > 0, "ask_book은 비어 있지 않은 문자열을 return 해야 합니다."
assert "<|im_start|>" not in book_answer and "<|im_end|>" not in book_answer, "답에 특수 토큰이 남았습니다. 새 토큰만 남기고 skip_special_tokens=True로 바꾸세요."
assert book_system not in book_answer, "답에 system 문장이 들어 있습니다. 입력 길이 뒤의 새 토큰만 글자로 바꾸세요."
assert "<think>" not in book_answer and "</think>" not in book_answer, "답에 추론 과정 표식(<think>, </think>)이 있습니다. ask_book 안의 apply_chat_template에 enable_thinking=False를 넣으세요."
assert ask_book(book_question) == book_answer, "같은 질문에 답이 달라집니다. ask_book 안에서 do_sample=False로 생성하세요."

assert isinstance(book_samples, list) and len(book_samples) == 3, "book_samples는 문자열 세 개의 리스트입니다."
assert all(isinstance(text, str) and len(text.strip()) > 0 for text in book_samples), "book_samples의 원소는 비어 있지 않은 문자열입니다."
assert len(set(book_samples)) >= 2, "세 답이 모두 같습니다. do_sample=True로 샘플링했는지, 시드를 for문 안에서 매번 고정하지 않았는지 확인하세요."
print("✅ 4번 통과")

**결과 읽기**
- **토큰 58개:** system·user 문장에 역할 표식(`<|im_start|>`, `<|im_end|>`)과 assistant 차례 표식, 빈 추론 칸(`<think>` `</think>`)이 붙은 길이입니다. 추론 모드를 켜 두면 끝이 `<think>`로 열린 채 끝나 56개가 되고, 모델은 그 뒤에 풀이 과정부터 씁니다. 5번에서 Ollama가 센 입력 토큰 수와 비교합니다.
- **greedy 답과 샘플링 답:** greedy는 다시 불러도 같고, 샘플링은 세 번의 문장이 서로 다릅니다. 모델은 이 서점의 배송 규정을 본 적이 없어 발송일 같은 사실을 그럴듯하게 지어낼 수 있습니다(문장은 실행 장치마다 다릅니다).

## 5. Ollama로 같은 질문과 모델 정보

**배경**: 같은 질문을 Ollama로 보내 생성 속도를 계산하고, 받은 모델 파일의 형식·양자화 수준·라이선스를 확인합니다(교안 01 4-1, 4-2).

**요구사항**
- (1) `ollama.chat`으로 `qwen3.5:4b-q4_K_M`에 `book_messages`를 보내 응답을 **`book_response`** 에 담습니다. 추론 모드는 끄고(`think=False`), 답한 뒤 바로 GPU에서 내리게 하고(`keep_alive=0`), `options`는 temperature 0, 새 토큰 최대 80개(`num_predict`)입니다.
- (1) 응답의 생성 토큰 수(`eval_count`)와 생성에 걸린 시간(`eval_duration`, 나노초)으로 토큰/초를 계산해 **`book_tokens_per_sec`**(실수)에 담습니다.
- (2) `ollama.show`로 `qwen3.5:4b-q4_K_M`의 정보를 **`book_show`** 에 담습니다.

**확인 기준**: 응답이 끝났고(`done`) 생성 토큰이 80개 이하이며, 추론 과정(`message.thinking`)이 비어 있습니다. 토큰/초가 1보다 크고 5000보다 작습니다. `book_show`의 파일 형식이 `gguf`, 양자화 수준이 `Q4_K_M`이고, 지원 기능에 `thinking`, `tools`, `vision`이 있으며 라이선스 원문에 `Apache`가 있습니다.

In [ ]:
# [작성] (1) Ollama로 같은 질문을 보내고 토큰/초를 계산하세요.
book_response = ...
book_tokens_per_sec = ...

print(book_response.message.content)
print("입력 토큰:", book_response.prompt_eval_count, "| 4번 입력 토큰:", book_prompt_tokens, "| 생성 토큰:", book_response.eval_count, "| 토큰/초:", round(book_tokens_per_sec, 1))

In [ ]:
# [작성] (2) Ollama가 가진 모델 정보를 받으세요.
book_show = ...

display(book_show.details)
print("지원 기능:", book_show.capabilities)
print(book_show.license[:120])

In [ ]:
# [자가채점] 응답 필드, 토큰/초, 모델 정보를 확인합니다.
assert book_response.done, "book_response는 ollama.chat의 응답입니다. 생성이 끝나지 않았습니다."
assert book_response.eval_count <= 80, "생성 토큰이 80개를 넘었습니다. options에 num_predict 80을 넣으세요."
assert not book_response.message.thinking, "응답에 추론 과정(message.thinking)이 있습니다. ollama.chat에 think=False를 넣어 추론 모드를 끄세요."
assert isinstance(book_tokens_per_sec, float) and 1 < book_tokens_per_sec < 5000, f"토큰/초 {book_tokens_per_sec}가 상식 범위(1~5000) 밖입니다. eval_duration의 단위(나노초)를 초로 바꿨는지 확인하세요."
assert book_show.details.format == "gguf", "book_show는 ollama.show('qwen3.5:4b-q4_K_M')의 결과입니다."
assert book_show.details.quantization_level == "Q4_K_M", "qwen3.5:4b-q4_K_M 태그의 양자화 수준은 Q4_K_M입니다. 다른 태그를 조회하지 않았는지 확인하세요."
assert "thinking" in book_show.capabilities and "tools" in book_show.capabilities and "vision" in book_show.capabilities, "qwen3.5:4b-q4_K_M의 지원 기능에는 추론 모드(thinking), 도구 호출(tools), 이미지 입력(vision)이 있습니다. book_show.capabilities를 확인하세요."
assert "Apache" in book_show.license, "라이선스 원문에 Apache가 없습니다. book_show.license를 확인하세요."
print("✅ 5번 통과")

**결과 읽기**
- **입력 토큰 수:** 4번에서 Transformers로 센 58개와 다를 수 있습니다. Ollama는 Hub의 chat template 파일 대신 Ollama 안에 만들어 둔 qwen3.5용 변환 규칙으로 메시지를 문자열로 바꾸므로, 역할 표식은 같아도 세부 문자열까지 같다는 보장이 없습니다.
- **토큰/초:** 장치와 그때 하는 다른 일에 따라 달라집니다. 이 호출처럼 처음 부르면 모델을 GPU에 올리고 준비하는 일이 겹쳐 생성 속도가 흔들릴 수 있어(불러오는 시간 자체는 `load_duration`에 따로 담깁니다), 속도를 비교할 때는 교안 02 7절처럼 첫 호출을 버리고 여러 번 측정합니다.
- **`gguf`, `Q4_K_M`, `thinking`, `tools`, `vision`, Apache:** `q4_K_M` 태그는 4비트 계열 GGUF 파일이고, 추론 모드·도구 호출·이미지 입력을 지원하며(6번의 도구 호출이 이 기능을 씁니다), Hub 모델 카드와 같은 Apache-2.0으로 배포됩니다. 이미지 처리 부분(projector)은 언어 모델 파일과 따로 담겨 함께 내려받습니다.

## 6. LangChain으로 구조화된 출력과 도구 호출

**배경**: 서점 접수 시스템은 문의를 정해진 분야와 긴급도로 받아야 하고, 배송 날짜 같은 사실은 모델이 지어내지 않게 주문 시스템을 조회해 답해야 합니다. 교안 01 4-4·4-5처럼 Ollama 모델을 LangChain `ChatOllama`로 씁니다.

**요구사항**
- (1) `ChatOllama`로 `qwen3.5:4b-q4_K_M`를 추론 모드를 끄고(`reasoning=False`) temperature 0으로 만들어 **`book_llm`** 에 담습니다. 제공한 `BookInquiry`로 구조화된 출력을 받게 해 `inquiry_messages`를 넣은 결과를 **`book_inquiry`** 에 담습니다.
- (2) 제공한 도구 `check_order_status`를 붙인 모델을 **`book_llm_with_tools`** 에 담고, `order_messages`를 넣은 응답을 **`book_ai_message`** 에 담습니다.
- (3) `book_ai_message`의 첫 도구 호출을 실행한 결과를 **`book_tool_result`** 에 담습니다. `order_messages`에 `book_ai_message`와 `ToolMessage`를 이어 붙이고 다시 물어 최종 응답을 **`book_final_message`** 에 담습니다.

**확인 기준**: `book_inquiry`가 `BookInquiry` 객체이고 요약이 비어 있지 않습니다. 도구 호출이 `check_order_status`를 `order_id` 인자로 부르고, 도구 결과와 최종 답이 비어 있지 않은 문자열입니다. 모델 답의 내용은 채점하지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01 4-4의 with_structured_output, 4-5의 bind_tools -> tool_calls -> 도구 실행 -> ToolMessage -> 다시 invoke 순서를 그대로 따릅니다.

세부구현:
1. 도구는 함수처럼 부르지 않고 도구.invoke(tool_call["args"])로 실행합니다.
2. ToolMessage에는 도구 결과와 함께 tool_call_id로 호출 번호를 넣습니다.
```

</details>

In [ ]:
from typing import Literal

from langchain_core.tools import tool
from pydantic import BaseModel


class BookInquiry(BaseModel):
    """서점 문의를 접수 시스템에 넣을 형식으로 분류한 결과"""
    category: Literal["배송", "환불", "재고", "결제"]
    urgency: Literal["낮음", "보통", "높음"]
    summary: str


@tool
def check_order_status(order_id: str) -> str:
    """주문번호로 서점 주문의 배송 상태를 조회합니다.

    Args:
        order_id: 주문번호(예: B1024)
    """
    # 실습용 주문 상태 테이블입니다. 실제 서비스에서는 서점 주문 시스템을 조회합니다
    order_status = {"B1024": "출고 대기(내일 오전 출고, 출고 다음 날 도착 예정)", "B2048": "배송 완료"}
    return order_status.get(order_id, "주문번호를 찾을 수 없음")


# 분류할 문의. 스키마의 키 뜻을 system에 함께 적어 둡니다
inquiry_messages = [
    {"role": "system", "content": "당신은 온라인 서점 문의 접수 담당입니다. 고객 문의를 category(분야), urgency(긴급도), summary(한 문장 요약)로 분류합니다."},
    {"role": "user", "content": "주문번호 B1024로 산 책이 아직 안 왔어요. 모레 선물해야 하는데 언제 받을 수 있나요?"},
]

# 주문 조회 도구로 답할 문의
order_messages = [
    {"role": "system", "content": book_system},
    {"role": "user", "content": "주문번호 B1024 책은 언제 받을 수 있나요?"},
]

In [ ]:
# [작성] (1) ChatOllama로 서점 문의를 BookInquiry 형식으로 분류하세요.
from langchain_ollama import ChatOllama

book_llm = ...
book_inquiry = ...
display(book_inquiry)

In [ ]:
# [작성] (2) 주문 조회 도구를 붙이고 도구 호출 요청을 받으세요.
book_llm_with_tools = ...
book_ai_message = ...
display(book_ai_message.tool_calls)

In [ ]:
# [작성] (3) 도구를 실행하고 결과를 돌려주어 최종 답을 받으세요.
from langchain_core.messages import ToolMessage

book_tool_result = ...
print("도구 결과:", book_tool_result)

book_final_message = ...
print(book_final_message.content)

In [ ]:
# [자가채점] 구조화된 출력의 형식, 도구 호출 요청, 최종 답을 확인합니다.
assert isinstance(book_inquiry, BookInquiry), "book_inquiry는 with_structured_output(BookInquiry)로 받은 BookInquiry 객체입니다."
assert len(book_inquiry.summary.strip()) > 0, "summary가 비어 있습니다."

assert len(book_ai_message.tool_calls) > 0, "book_ai_message에 도구 호출 요청이 없습니다. bind_tools로 도구를 붙인 모델에 order_messages를 넣었는지 확인하세요."
assert book_ai_message.tool_calls[0]["name"] == "check_order_status", "도구 호출 이름이 check_order_status가 아닙니다."
assert "order_id" in book_ai_message.tool_calls[0]["args"], "도구 호출 인자에 order_id가 없습니다."

assert isinstance(book_tool_result, str) and len(book_tool_result) > 0, "book_tool_result는 도구를 실행한 결과 문자열입니다. 도구.invoke(인자)로 실행하세요."
assert isinstance(order_messages[-1], ToolMessage), "order_messages 끝에 ToolMessage를 붙였는지 확인하세요."
assert isinstance(book_final_message.content, str) and len(book_final_message.content.strip()) > 0, "book_final_message는 도구 결과를 받은 뒤 다시 invoke 한 최종 응답입니다."
print("✅ 6번 통과")

**결과 읽기**
- **구조화된 출력:** 세 키가 모두 있고 분야와 긴급도가 허용 값 안에 있어 접수 시스템의 열에 바로 넣을 수 있습니다. 분류가 알맞은지는 사람이 확인합니다.
- **도구 호출:** 모델이 질문의 주문번호를 `order_id` 인자로 넣었는지, 최종 답이 4번처럼 날짜를 지어내지 않고 도구 결과(내일 오전 출고)를 근거로 안내하는지 봅니다.

## 판단 쓰기

1~6번의 결과를 보고, 서점에 어떤 모델을 권할지와 그 이유, 이 답을 고객에게 그대로 보내면 안 되는 이유를 두세 문장으로 씁니다.

**판단** (두세 문장)

- ...